# LLM 직접 불러 보기 · OpenAI 형식

강의 12 ~ 17장에서 본 것을 실제 호출로 확인한다.

- 실행 환경: Jupyter + Deno 커널 (TypeScript). 준비 방법은 `README.md`
- 기본값은 이 PC 의 로컬 Ollama. OpenAI 로 바꾸려면 아래 첫 칸의 세 값만 바꾼다

In [ ]:
// 접속 정보: 이 세 값만 바꾸면 OpenAI, Ollama, vLLM 어디든 같은 코드로 돈다
const BASE_URL = Deno.env.get("LLM_BASE_URL") ?? "http://localhost:11434/v1";  // OpenAI: https://api.openai.com/v1
const API_KEY  = Deno.env.get("LLM_API_KEY")  ?? "ollama";                     // Ollama 는 아무 값이나 된다
const MODEL    = Deno.env.get("LLM_MODEL")    ?? "qwen3.5:9b-q4_K_M";

// qwen3.5 처럼 답하기 전에 "생각" 부터 하는 모델은, 시연에서 답만 바로 보려고 생각 단계를 끈다
// 이 옵션을 받지 않는 모델(예: OpenAI gpt-4o 계열)이면 {} 로 바꾼다
const OPTIONS = { reasoning_effort: "none" };

## ① LLM 호출은 HTTP 요청 하나다

LLM 을 부르는 건 `/chat/completions` 주소로 POST 요청을 한 번 보내는 것이다. 요청에 담는 건 두 가지다.

- `model`: 어떤 모델에게 물을지
- `messages`: 지금까지의 대화. 메시지 하나는 `{ role, content }` 한 쌍이다

| role | 누가 한 말인가 | 예 |
| --- | --- | --- |
| `system` | 개발자가 정하는 역할 · 규칙. 대화 맨 앞에 둔다 | "너는 교육장 일정 담당이다." |
| `user` | 사용자가 한 말 | "29일에 제2교육장 비어 있어?" |
| `assistant` | 모델이 한 말. 지난 답을 다시 보낼 때 쓴다 | "16시 이후가 비어 있습니다." |

`content` 는 그 말의 내용이다. 그냥 글자다.

In [ ]:
const res = await fetch(`${BASE_URL}/chat/completions`, {
  method: "POST",
  headers: { "Content-Type": "application/json", Authorization: `Bearer ${API_KEY}` },
  body: JSON.stringify({
    model: MODEL,                                                // 어떤 모델에게
    messages: [                                                  // 지금까지의 대화
      { role: "system", content: "너는 교육장 일정 담당이다." },     // 역할 · 규칙
      { role: "user", content: "29일에 제2교육장 비어 있어?" },     // 사용자의 말
    ],
    ...OPTIONS,                                                  // 위에서 정한 옵션
  }),
});
const data = await res.json();
console.log(data.choices[0].message);

답도 같은 모양으로 돌아온다. `choices[0].message` 에 `{ role: "assistant", content }` 메시지 하나가 들어 있다.

응답 전체를 보면 답 말고도 `usage` 에 이번 호출에 쓴 토큰 수가 들어 있다.

In [ ]:
console.log(JSON.stringify(data, null, 2));

## ② 대화는 결국 긴 messages 배열이다 (15장)

LLM 에게 "대화" 는 따로 없다. 매번 보내는 `messages` 배열이 전부다. 앞으로는 아래 `chat()` 으로 부른다. 하는 일은 위의 `fetch` 와 똑같다.

In [ ]:
type Message = { role: "system" | "user" | "assistant"; content: string };

async function chat(messages: Message[], extra: Record<string, unknown> = {}) {
  const res = await fetch(`${BASE_URL}/chat/completions`, {
    method: "POST",
    headers: { "Content-Type": "application/json", Authorization: `Bearer ${API_KEY}` },
    body: JSON.stringify({ model: MODEL, messages, ...OPTIONS, ...extra }),
  });
  if (!res.ok) throw new Error(`${res.status} ${await res.text()}`);
  return await res.json();
}

보내는 배열을 눈으로 보려고, 배열을 한 줄씩 찍어 주는 함수도 하나 만든다.

In [ ]:
function show(messages: Message[]) {
  for (const m of messages) console.log(`  ${m.role.padEnd(9)} │ ${m.content}`);
}

### 첫 번째 호출

In [ ]:
const messages: Message[] = [
  { role: "system", content: "너는 교육장 일정 담당이다. 짧게 답한다." },
  { role: "user", content: "29일에 제2교육장에서 교육 잡기 좋은 시간을 하나만 추천해 줘" },
];
show(messages);

const first = await chat(messages);
const firstAnswer = first.choices[0].message;  // { role: "assistant", content: "..." }
console.log("\n답:", firstAnswer.content);

### 두 번째 호출 · 답을 배열에 넣지 않으면

질문만 이어 붙여 보낸다. 방금 받은 답은 배열에 없다. 첫 번째 답과 비교해 보자.

In [ ]:
const withoutAnswer: Message[] = [
  ...messages,
  { role: "user", content: "방금 추천한 시간을 다시 말해 줘" },
];
show(withoutAnswer);

const second = await chat(withoutAnswer);
console.log("\n답:", second.choices[0].message.content);  // 첫 번째 답을 모른다. 모른다고 하거나, 새로 지어낸다

### 두 번째 호출 · 답을 배열에 넣으면

첫 번째 답을 `assistant` 메시지로 넣고 나서 질문을 붙인다.

In [ ]:
const withAnswer: Message[] = [
  ...messages,
  firstAnswer,                                              // 받은 답을 그대로 넣는다
  { role: "user", content: "방금 추천한 시간을 다시 말해 줘" },
];
show(withAnswer);

const third = await chat(withAnswer);
console.log("\n답:", third.choices[0].message.content);

LLM 이 기억하는 게 아니다. 내가 배열에 넣어서 다시 보낸 만큼만 안다. **대화를 이어 가는 건 이 배열을 이어 붙이는 일**이다.

## ③ 대화가 길어질수록 보내는 토큰이 쌓인다 (16 ~ 17장)

질문할 때마다 지난 대화를 통째로 다시 보낸다. 입력 토큰 수가 계속 늘어나는 걸 보자.

In [ ]:
const history: Message[] = [
  { role: "system", content: "너는 교육장 일정 담당이다. 한두 문장으로 짧게 답한다." },
];

async function ask(text: string) {
  history.push({ role: "user", content: text });
  const r = await chat(history);
  const reply = r.choices[0].message.content;
  history.push({ role: "assistant", content: reply });
  console.log(`보낸 메시지 ${history.length - 1}개 · 입력 토큰 ${r.usage.prompt_tokens}\n  → ${reply}\n`);
}

await ask("29일에 제2교육장 비어 있어?");
await ask("그럼 오후 4시로 잡아 줘");
await ask("참석자는 20명이야. 의자 충분해?");
await ask("지금까지 잡은 일정을 정리해 줘");

## ④ 컨텍스트 윈도우를 넘기면 앞부분이 잘린다 (17장)

맨 앞에 암호를 숨기고, 의미 없는 메모로 글을 늘린 뒤, 맨 끝에서 암호를 묻는다. 이 노트북의 Ollama 는 컨텍스트 윈도우를 **4,096 토큰**으로 띄운다.

In [ ]:
async function secretTest(repeat: number) {
  const filler = "교육장 일정표 메모: 오늘은 특별한 일정이 없다. ".repeat(repeat);
  const r = await chat([
    { role: "system", content: "너는 교육장 일정 담당이다. 짧게 답한다." },
    { role: "user", content: `암호는 '파란 고래'다. 꼭 기억해. ${filler}\n\n맨 처음에 말한 암호가 뭐였지?` },
  ]);
  console.log(`메모 ${repeat}번 · 실제로 들어간 입력 토큰 ${r.usage.prompt_tokens}`);
  console.log(`  → ${r.choices[0].message.content}\n`);
}

await secretTest(100);  // 한도 안: 약 1,300 토큰
await secretTest(400);  // 한도 밖: 약 4,900 토큰을 보낸다

- 메모 100번은 한도 안이라 암호를 기억한다
- 메모 400번은 약 4,900 토큰을 보냈는데, 실제로 들어간 건 2,000 토큰 남짓이다. 앞부분이 **오류 없이 조용히 잘려서** 암호가 사라졌다. 모델은 잘린 자리의 토막 난 글자("너표 메모" 같은)를 맨 처음이라고 답하기도 한다
- 서버마다 다르다. Ollama 는 이렇게 잘라 버리고, OpenAI 는 `context_length_exceeded` 오류를 돌려준다. 어느 쪽이든 무엇을 남기고 뺄지는 하네스가 정해야 한다

## 정리

- LLM 호출은 `messages` 를 담은 요청 하나다
- 대화는 이 `messages` 배열을 이어 붙인 것이다. 답을 넣지 않으면 다음 호출은 그 답을 모른다
- 배열이 컨텍스트 윈도우를 넘으면 앞부분이 잘린다. 무엇을 넣고 뺄지 정하는 게 하네스의 일이다
- 다음: 여기에 도구를 붙인다